## Проверка SMOTE-версии модели на тестовом датасете ##

## 01 Импорт данных ##

In [1]:
import pandas as pd
import numpy as np
import pickle
import joblib
import re
import pymorphy3
from nltk.corpus import stopwords
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, 
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
print("\n" + "="*80)
print("🧪 ПРОВЕРКА МОДЕЛИ НА ТЕСТОВОМ ДАТАСЕТЕ")
print("="*80)

# === ЗАГРУЗКА КОМПОНЕНТОВ ===
print("\n1️⃣ Загрузка компонентов...")

# Загружаем улучшенную модель (SMOTE)
classifier = joblib.load('../models/v1.0/classifier_smote.pkl')
vectorizer = joblib.load('../models/v1.0/vectorizer.pkl')
label_encoder = joblib.load('../models/v1.0/label_encoder.pkl')

print("   ✓ classifier_smote.pkl загружен")
print("   ✓ vectorizer.pkl загружен")
print("   ✓ label_encoder.pkl загружен")

# === ЗАГРУЗКА ТЕСТОВЫХ ДАННЫХ ===
print("\n2️⃣ Загрузка тестовых данных...")

test_df = pd.read_csv('../data/service_desk_test.csv', sep=';', encoding='utf-8')

print(f"   ✓ Загружено {len(test_df)} примеров")
print(f"   ✓ Колонки: {list(test_df.columns)}")

# Проверяем, есть ли целевая переменная
if 'Тип задачи' in test_df.columns:
    print("   ✓ Найдена целевая переменная 'Тип задачи'")
    has_target = True
else:
    print("   ✗ Целевая переменная НЕ найдена")
    has_target = False

print(f"   ✓ Примеры в датасете: {len(test_df)}")


🧪 ПРОВЕРКА МОДЕЛИ НА ТЕСТОВОМ ДАТАСЕТЕ

1️⃣ Загрузка компонентов...
   ✓ classifier_smote.pkl загружен
   ✓ vectorizer.pkl загружен
   ✓ label_encoder.pkl загружен

2️⃣ Загрузка тестовых данных...
   ✓ Загружено 1000 примеров
   ✓ Колонки: ['id', 'Тип задачи', 'Тема', 'Срок исполнения']
   ✓ Найдена целевая переменная 'Тип задачи'
   ✓ Примеры в датасете: 1000


## 02 Определение TextPreprocessor ##

In [5]:
print("\n3️⃣ Инициализация TextPreprocessor...")

class TextPreprocessor:
    """Класс для предобработки текста"""
    
    def __init__(self):
        self.morph = pymorphy3.MorphAnalyzer()
        self.russian_stopwords = set(stopwords.words('russian'))
    
    def clean_text(self, text):
        if pd.isna(text):
            return ""
        text = str(text).lower()
        text = re.sub(r'\S+@\S+', ' ', text)
        text = re.sub(r'http\S+|www\S+', ' ', text)
        text = re.sub(r'\d{1,2}[./\-]\d{1,2}[./\-]\d{2,4}', ' DATE ', text)
        text = re.sub(r'\d+', ' NUM ', text)
        text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    def lemmatize(self, text):
        if not text:
            return ""
        words = text.split()
        lemmatized_words = []
        for word in words:
            if word not in self.russian_stopwords and len(word) > 2:
                parsed = self.morph.parse(word)[0]
                lemma = parsed.normal_form
                lemmatized_words.append(lemma)
        return ' '.join(lemmatized_words)
    
    def preprocess(self, text):
        text = self.clean_text(text)
        text = self.lemmatize(text)
        return text

preprocessor = TextPreprocessor()
print("   ✓ TextPreprocessor инициализирован")


3️⃣ Инициализация TextPreprocessor...
   ✓ TextPreprocessor инициализирован


## 03 Предобработка тестовых данных ##

In [7]:
print("\n4️⃣ Предобработка текстов...")

# Подготавливаем текст
test_df['text'] = test_df['Тема'].fillna('')
test_df['processed_text'] = test_df['text'].apply(preprocessor.preprocess)

print(f"   ✓ Обработано {len(test_df)} текстов")
print(f"   ✓ Примеры обработанных текстов:")

for i in range(min(3, len(test_df))):
    original = test_df['text'].iloc[i][:60]
    processed = test_df['processed_text'].iloc[i][:60]
    print(f"      Оригинал:  {original}...")
    print(f"      Обработано: {processed}...")
    print()


4️⃣ Предобработка текстов...
   ✓ Обработано 1000 текстов
   ✓ Примеры обработанных текстов:
      Оригинал:  HR. Долги по обучению, аккаунт BFTraining - Комиссаров Вячес...
      Обработано: долг обучение аккаунт bftraining комиссар вячеслав дмитриеви...

      Оригинал:  ДТиОР. Передать списки во FriendWork - Комиссаров Вячеслав Д...
      Обработано: дтиор передать список friendwork комиссар вячеслав дмитриеви...

      Оригинал:  ДТиОР. Установить замещение в СЭД - Комиссаров Вячеслав Дмит...
      Обработано: дтиор установить замещение сэд комиссар вячеслав дмитриевич...



## 04 Векторизация ## 

In [12]:
print("\n" + "="*80)
print("🔄 ПЕРЕСОЗДАНИЕ SMOTE МОДЕЛИ (ПОЛНОСТЬЮ)")
print("="*80)

import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import re
import pymorphy3
from nltk.corpus import stopwords

print("\n1️⃣ Загрузка исходных данных...")

# Загружаем исходные данные
train_df = pd.read_csv('../data/service_desk_train.csv', sep=';', encoding='utf-8')

print(f"   ✓ Train датасет: {len(train_df)} примеров")

print("\n2️⃣ Определение TextPreprocessor...")

class TextPreprocessor:
    def __init__(self):
        self.morph = pymorphy3.MorphAnalyzer()
        self.russian_stopwords = set(stopwords.words('russian'))
    
    def clean_text(self, text):
        if pd.isna(text):
            return ""
        text = str(text).lower()
        text = re.sub(r'\S+@\S+', ' ', text)
        text = re.sub(r'http\S+|www\S+', ' ', text)
        text = re.sub(r'\d{1,2}[./\-]\d{1,2}[./\-]\d{2,4}', ' DATE ', text)
        text = re.sub(r'\d+', ' NUM ', text)
        text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    def lemmatize(self, text):
        if not text:
            return ""
        words = text.split()
        lemmatized_words = []
        for word in words:
            if word not in self.russian_stopwords and len(word) > 2:
                parsed = self.morph.parse(word)[0]
                lemma = parsed.normal_form
                lemmatized_words.append(lemma)
        return ' '.join(lemmatized_words)
    
    def preprocess(self, text):
        text = self.clean_text(text)
        text = self.lemmatize(text)
        return text

preprocessor = TextPreprocessor()
print("   ✓ TextPreprocessor готов")

print("\n3️⃣ Предобработка текстов...")

# Обрабатываем
train_df['text'] = train_df['Тема'].fillna('')
train_df['processed_text'] = train_df['text'].apply(preprocessor.preprocess)
train_df['processed_text'] = train_df['processed_text'].fillna('')  # Заполняем NaN

print(f"   ✓ Обработано {len(train_df)} текстов")
print(f"   ✓ Пропусков: {train_df['processed_text'].isna().sum()}")

print("\n4️⃣ Кодирование целевой переменной...")

label_encoder = LabelEncoder()
train_df['target_encoded'] = label_encoder.fit_transform(train_df['Тип задачи'])

print(f"   ✓ Классов: {len(label_encoder.classes_)}")

print("\n5️⃣ Создание vectorizer...")

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

# Обучаем на БЕЗ NaN данных
vectorizer.fit(train_df['processed_text'])

X = vectorizer.transform(train_df['processed_text'])
y = train_df['target_encoded'].values

num_features = X.shape[1]
print(f"   ✓ Vectorizer создан")
print(f"   ✓ Матрица X: {X.shape}")
print(f"   ✓ Признаков: {num_features}")

print("\n6️⃣ Train/Val split...")

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   ✓ Train: {X_train.shape[0]} примеров")
print(f"   ✓ Val: {X_val.shape[0]} примеров")

print("\n7️⃣ Применение SMOTE...")

smote = SMOTE(k_neighbors=2, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"   ✓ До SMOTE: {len(y_train)} примеров")
print(f"   ✓ После SMOTE: {len(y_train_balanced)} примеров")

print("\n8️⃣ Обучение SMOTE модели...")

classifier_smote = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    verbose=0
)

classifier_smote.fit(X_train_balanced, y_train_balanced)

print(f"   ✓ Модель обучена")
print(f"   ✓ Ожидает признаков: {classifier_smote.n_features_in_}")
print(f"   ✓ Vectorizer имеет: {X_train.shape[1]} признаков")

print("\n9️⃣ Проверка совместимости...")

if classifier_smote.n_features_in_ == X_train.shape[1]:
    print(f"   ✅ СОВМЕСТИМО!")
else:
    print(f"   ❌ ОШИБКА: Несовместимо")

print("\n🔟 Оценка на validation...")

from sklearn.metrics import accuracy_score, f1_score

y_val_pred = classifier_smote.predict(X_val)

val_acc = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)

print(f"   ✓ Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")
print(f"   ✓ F1-score: {val_f1:.4f}")

print("\n1️⃣1️⃣ Сохранение артефактов...")

# Сохраняем все артефакты
joblib.dump(classifier_smote, '../models/v1.0/classifier_smote_new.pkl')
joblib.dump(vectorizer, '../models/v1.0/vectorizer_smote.pkl')
joblib.dump(label_encoder, '../models/v1.0/label_encoder_smote.pkl')

print(f"   ✓ classifier_smote_new.pkl")
print(f"   ✓ vectorizer_smote.pkl")
print(f"   ✓ label_encoder_smote.pkl")

print("\n" + "="*80)
print("✅ СМОТЭ МОДЕЛЬ ПЕРЕСОЗДАНА И СОХРАНЕНА")
print("="*80)

# Теперь используем эту модель для тестирования
print("\n1️⃣2️⃣ Тестирование на test датасете...")

test_df = pd.read_csv('../data/service_desk_test.csv', sep=';', encoding='utf-8')

print(f"   ✓ Test датасет: {len(test_df)} примеров")

# Обрабатываем test
test_df['text'] = test_df['Тема'].fillna('')
test_df['processed_text'] = test_df['text'].apply(preprocessor.preprocess)

# Векторизируем
X_test = vectorizer.transform(test_df['processed_text'])

print(f"   ✓ X_test: {X_test.shape}")
print(f"   ✓ Совместимо: {X_test.shape[1] == classifier_smote.n_features_in_}")

# Предсказываем
y_test_pred = classifier_smote.predict(X_test)
y_test_proba = classifier_smote.predict_proba(X_test)

test_df['predicted_type'] = label_encoder.inverse_transform(y_test_pred)
test_df['confidence'] = np.max(y_test_proba, axis=1)

print(f"   ✓ Получены предсказания")

# Если есть правильные ответы
if 'Тип задачи' in test_df.columns:
    y_test_true = label_encoder.transform(test_df['Тип задачи'])
    test_acc = accuracy_score(y_test_true, y_test_pred)
    test_f1 = f1_score(y_test_true, y_test_pred, average='weighted', zero_division=0)
    
    print(f"\n" + "="*80)
    print("📈 РЕЗУЛЬТАТЫ НА ТЕСТОВОМ ДАТАСЕТЕ")
    print("="*80)
    print(f"\n   Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"   Test F1-score: {test_f1:.4f}")
    
    # Сравнение
    print(f"\n📊 СРАВНЕНИЕ МЕТРИК:")
    print(f"   Validation Accuracy: {val_acc*100:.2f}%")
    print(f"   Test Accuracy: {test_acc*100:.2f}%")
    print(f"   Разница: {abs(val_acc - test_acc)*100:.2f}%")

print("\n✅ ГОТОВО!")



🔄 ПЕРЕСОЗДАНИЕ SMOTE МОДЕЛИ (ПОЛНОСТЬЮ)

1️⃣ Загрузка исходных данных...
   ✓ Train датасет: 8565 примеров

2️⃣ Определение TextPreprocessor...
   ✓ TextPreprocessor готов

3️⃣ Предобработка текстов...
   ✓ Обработано 8565 текстов
   ✓ Пропусков: 0

4️⃣ Кодирование целевой переменной...
   ✓ Классов: 17

5️⃣ Создание vectorizer...
   ✓ Vectorizer создан
   ✓ Матрица X: (8565, 4402)
   ✓ Признаков: 4402

6️⃣ Train/Val split...
   ✓ Train: 6852 примеров
   ✓ Val: 1713 примеров

7️⃣ Применение SMOTE...
   ✓ До SMOTE: 6852 примеров
   ✓ После SMOTE: 64447 примеров

8️⃣ Обучение SMOTE модели...
   ✓ Модель обучена
   ✓ Ожидает признаков: 4402
   ✓ Vectorizer имеет: 4402 признаков

9️⃣ Проверка совместимости...
   ✅ СОВМЕСТИМО!

🔟 Оценка на validation...
   ✓ Accuracy: 0.9819 (98.19%)
   ✓ F1-score: 0.9826

1️⃣1️⃣ Сохранение артефактов...
   ✓ classifier_smote_new.pkl
   ✓ vectorizer_smote.pkl
   ✓ label_encoder_smote.pkl

✅ СМОТЭ МОДЕЛЬ ПЕРЕСОЗДАНА И СОХРАНЕНА

1️⃣2️⃣ Тестирование на test 